# agentic-rl-wordle — 救援用：評測 + push HF（不重新訓練）

用途：主 notebook（wordle_grpo_colab_train.ipynb）的訓練已完成（checkpoint 都在 Drive），
但最後的「評測 + push HF」段沒有完成時，用這本獨立補跑 M3.4/M3.5。

- GPU 選 **L4 即可**（純推理，1.5B 模型綽綽有餘，比 A100 便宜 3 倍以上）
- 直接「全部執行」；結束會自動把 `final_report.md` 印在 cell 輸出裡並釋放機器
- 前提：Drive 上已有 `wordle_rl_bundle.zip` 與 `runs/<RUN_NAME>/` 的訓練產物

In [ ]:
# ===== ① 參數 =====
RUN_NAME = "wordle-grpo-v2"       # 要評測的那次訓練的資料夾名
HF_USERNAME = "steven0226"
DRIVE_BASE = "/content/drive/MyDrive/agentic-rl-wordle"

In [ ]:
# ===== ② Drive + bundle + 依賴（與主 notebook cell② 相同）=====
from google.colab import drive
drive.mount('/content/drive')

import pathlib
BUNDLE = f"{DRIVE_BASE}/wordle_rl_bundle.zip"
assert pathlib.Path(BUNDLE).exists(), f"找不到 {BUNDLE}"

!rm -rf /content/agentic-rl-wordle && mkdir -p /content/agentic-rl-wordle
!unzip -q -o "$BUNDLE" -d /content/agentic-rl-wordle
%cd /content/agentic-rl-wordle

!pip install -q -e .
!pip install -q -r requirements-colab.txt

import trl, vllm
print("trl", trl.__version__, "| vllm", vllm.__version__)

!python scripts/fetch_words.py

In [ ]:
# ===== ③ HF_TOKEN =====
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN OK")

In [ ]:
# ===== ④ 盤點訓練產物 → 遞迴找 adapter → 評測 → 印 final_report → push HF → 釋放機器 =====
# ⚠️ 子行程直接繼承 stdout 時 Colab 常常什麼都不顯示（主 notebook cell⑤/⑥ 的同款教訓，
#    也是主 notebook 評測段「看起來沒反應」的原因）：一律寫 log 檔再自己 print。
# ⚠️ adapter 用 rglob 遞迴搜尋：實測 v2 run 的 final/ 沒有落地（原因待查——train.py 的
#    save_model 有跑完（rc=0、README.md 有寫出）但 final/ 不在 Drive 上），而第一版只檢查
#    checkpoint-* 最外層也沒找到 adapter_config.json——遞迴搜尋不管 Trainer 存在哪一層都能找到。
import pathlib
import shutil
import subprocess
import sys

CKPT_DIR = f"{DRIVE_BASE}/runs/{RUN_NAME}"
run_dir = pathlib.Path(CKPT_DIR)

print("=== run 目錄盤點 ===", flush=True)
for p in sorted(run_dir.iterdir()):
    if p.is_dir():
        print(" ", p.name + "/", flush=True)
        if p.name != "samples":
            for q in sorted(p.iterdir())[:25]:
                size = "" if q.is_dir() else f"（{q.stat().st_size:,} bytes）"
                print("    ", q.name, size, flush=True)
    else:
        print(" ", p.name, f"（{p.stat().st_size:,} bytes）", flush=True)

cfgs = sorted(run_dir.rglob("adapter_config.json"))
print("\n找到的 adapter：", [str(c.parent) for c in cfgs], flush=True)
# 故意放在 try/finally 之外：找不到 adapter 時保持機器連線，方便接著人工診斷
assert cfgs, "整個 run 目錄都沒有 adapter 檔——把上面的盤點結果整段貼回去分析"


def _prio(c):
    name = c.parent.name
    if name == "final":
        return float("inf")
    try:
        return int(name.split("-")[-1])
    except ValueError:
        return -1

adapter = max(cfgs, key=_prio).parent
print("使用 adapter:", adapter, flush=True)


def run_logged(cmd, log_name):
    log_path = f"{CKPT_DIR}/{log_name}"
    print(">>>", " ".join(map(str, cmd)), "| log:", log_path, flush=True)
    with open(log_path, "ab") as f:
        rc = subprocess.call([str(c) for c in cmd], stdout=f, stderr=subprocess.STDOUT)
    text = pathlib.Path(log_path).read_text(errors="ignore")
    print("\n".join(text.splitlines()[-40:]), flush=True)
    assert rc == 0, f"失敗（returncode={rc}），完整 log：{log_path}"


try:
    run_logged([sys.executable, "eval/run_eval.py",
                "--adapter", adapter, "--backend", "vllm"], "eval.log")
    shutil.copytree("results", f"{CKPT_DIR}/results", dirs_exist_ok=True)

    report = pathlib.Path("results/final_report.md")
    if report.exists():
        print("\n" + "=" * 28 + " final_report.md " + "=" * 28 + "\n", flush=True)
        print(report.read_text(encoding="utf-8"), flush=True)
    else:
        print("⚠ results/final_report.md 不存在，results/ 內容：",
              sorted(p.name for p in pathlib.Path("results").glob("*")), flush=True)

    run_logged([sys.executable, "scripts/push_model.py",
                "--adapter", adapter,
                "--repo", f"{HF_USERNAME}/qwen2.5-1.5b-wordle-grpo",
                "--card", "docs/model_card.md"], "push.log")
    print("\nHF push 完成 ✅  → https://huggingface.co/" + HF_USERNAME + "/qwen2.5-1.5b-wordle-grpo", flush=True)
finally:
    from google.colab import runtime
    runtime.unassign()